# Propensity score weighing for outcomes in wake-up stroke treatment (ESOC21)

In [ ]:
import pandas as pd
from causalinference import CausalModel
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from causalinference.estimators.matching import match
data_path = './data/ESOC21/bridg_vs_EVT_HUG_CHUV_ESOC_only CT.xlsx'

## Outcome variables:
any_bleeding infarct_growth	mRS_score_0-1_at_90_days  TICI_2b-3 Symptomatic_intracranial_haemorrhage Parenchymal_haematoma_type_2
NIHSS_score_at_24_h
Change_in_NIHSS_score_from_baseline_to_24_h
8_points_NIHSS_reduction_at_24_h
Change_in_NIHSS_score_from_baseline_to_24_h
mRS_shift_at_90_days
mRS_0-2_at_90_days
Death_at_7_days
Death_at_90_days

In [ ]:
outcome_var = 'Parenchymal_haematoma_type_2'

## Covariates

In [ ]:
covar_fields = ['Age', 'Sex', 'Time_from_symtoms_recognition_to_groin_min', 'Time_from_LSW_to_groin_min',
                'NIHSS_on_admission',
                'Med_Hist_Hypertension', 'Med Hist Diabetes', 'Med Hist Hyperlipidemia',
                'Med Hist Atrial Fibr', 'Wake-up stroke', 'Hypoperfused tissue volume (Tmax > 6s) ml',
                'Ischemic core volume (CBF < 30%) ml', 'Site of occlusion', 'ASPECTS score (CT or MRI)']

if outcome_var == 'infarct_growth':
    # 9 covariates for infarct growth
    covar_fields = [ 'Time_from_symtoms_recognition_to_groin_min', 'Time_from_LSW_to_groin_min',  'NIHSS_on_admission',
                     'Wake-up stroke',
                     'Hypoperfused tissue volume (Tmax > 6s) ml', 'Ischemic core volume (CBF < 30%) ml',
                     'Site of occlusion', 'ASPECTS score (CT or MRI)', 'Age']


## Intervention : IVT with rTPA

In [ ]:
intervention_var = 'IVT_with_rtPA'

In [ ]:

data = pd.read_excel(data_path)

data.columns = data.columns.str.strip()
data.columns = data.columns.str.replace(' ','_')
covar_fields = [covar.replace(' ','_') for covar in covar_fields]

# Encode mRS as int
data['mRS_score_0-1_at_90_days'] = data['mRS_score_0-1_at_90_days'].astype('Int64')

# Encode wake-up
data['Wake-up_stroke'] = data['Wake-up_stroke'].astype('int')

# Encode infarct growth
data['infarct_growth'] = data['Lesion_volume_(ml)'] - data['Ischemic_core_volume_(CBF_<_30%)_ml']
data.loc[data['infarct_growth'] < 0, 'infarct_growth'] = 0

# Encode change in mRS
data['mRS_shift_at_90_days'] = data['mRS_score_at_90_days'] - data['Pre-stroke_mRS']

# Encode TICI
data['TICI_2b-3'] = data['TICI_2b-3'].astype('int')

# Retain only relevant variables
data = data[covar_fields + [outcome_var, intervention_var]]

In [ ]:
# Number treated
n_t = data[intervention_var].sum()

# Number not treated
n_c = data[intervention_var].size - data[intervention_var].sum()
print('Before dropping NA', n_t, n_c)

In [ ]:
# Number of events in treatment group
n_events_t = data[data[intervention_var] == 1][outcome_var].sum()
print('Before dropping NA', n_events_t, n_events_t/n_t)

# Number of events in control group
n_events_c =data[data[intervention_var] == 0][outcome_var].sum()
print('Before dropping NA', n_events_c, n_events_c/n_c)

In [ ]:
# median and IQR of events in treatment group
print(data[data[intervention_var] == 1][outcome_var].median())
data[data[intervention_var] == 1][outcome_var].quantile(0.25), data[data[intervention_var] == 1][outcome_var].quantile(0.75)

In [ ]:
# median and IQR of events in control group
print(data[data[intervention_var] == 0][outcome_var].median())
data[data[intervention_var] == 0][outcome_var].quantile(0.25), data[data[intervention_var] == 0][outcome_var].quantile(0.75)


In [ ]:
# Drop NaN (has to be done before one-hot-encoding)
data = data.dropna()

In [ ]:
# Clean localisation column
if 'Site_of_occlusion' in data.columns:
    data['Site_of_occlusion'] = data['Site_of_occlusion'].str.replace('ICA','ACI')
    data['Site_of_occlusion'] = data['Site_of_occlusion'].str.replace('et','+')

# One-hot encoding for categorical vars
if 'Sex' in data.columns: data['Sex'] = data['Sex'] == 1
if 'Site_of_occlusion' in data.columns:
    data['Site_of_occlusion'] = pd.Categorical(data['Site_of_occlusion'])
    data['Site_of_occlusion'] = data['Site_of_occlusion'].cat.codes

data = data.astype(float)


In [ ]:
# Number treated
n_t = data[intervention_var].sum()
# Number not treated
n_c = data[intervention_var].size - data[intervention_var].sum()
print('After dropping NA', n_t, n_c)

In [ ]:
# Number of events in treatment group
n_events_t = data[data[intervention_var] == 1][outcome_var].sum()
print('After dropping NA', n_events_t, n_events_t/n_t)

# Number of events in control group
n_events_c =data[data[intervention_var] == 0][outcome_var].sum()
print('After dropping NA', n_events_c, n_events_c/n_c)

In [ ]:
# median and IQR of events in treatment group
print(data[data[intervention_var] == 1][outcome_var].median())
data[data[intervention_var] == 1][outcome_var].quantile(0.25), data[data[intervention_var] == 1][outcome_var].quantile(0.75)

In [ ]:
# median and IQR of events in control group
print(data[data[intervention_var] == 0][outcome_var].median())
data[data[intervention_var] == 0][outcome_var].quantile(0.25), data[data[intervention_var] == 0][outcome_var].quantile(0.75)


Defining model with regards to IVT as intervention

In [ ]:
intervention = data.IVT_with_rtPA.to_numpy()
outcome = data[outcome_var].to_numpy()

covars = data[covar_fields].to_numpy()

In [ ]:
causal = CausalModel(outcome, intervention, covars)
print(causal.summary_stats)

Estimate propensity scores

In [ ]:
causal.est_propensity()
print(causal.propensity)
print(causal.propensity.keys())

Match by nearest neighbour defined as distance between the covariate vectors

In [ ]:
# causal.est_via_matching(bias_adj=True)
# print(f'Outcome variable: {outcome_var}')
# print(causal.estimates)


Match by blocks of propensity scores

In [ ]:
# causal.stratify_s()
# print(causal.strata)
# causal.est_via_blocking()
# print(f'Outcome variable: {outcome_var}')
# print(causal.estimates)


Match through log regression of covariates and propensity score (Horvitz-Thompson weighting estimator)

In [ ]:
causal.est_via_weighting()
print(f'Outcome variable: {outcome_var}')
print(causal.estimates)

In [ ]:
data['propensity_score'] = causal.propensity['fitted']

## Distribution of covariates with respect to treatment and outcome

In [ ]:
g = sns.pairplot(
    data,
    x_vars=covar_fields,
    y_vars=[outcome_var],
    hue=intervention_var,
    kind='scatter',
)
g.map(sns.kdeplot, alpha=.5, levels=4)

g.fig.suptitle(f'Distribution of covariates with respect to treatment and outcome ({outcome_var})', y=1.08)

plt.plot()

In [ ]:
g.savefig(f'{outcome_var}_covariates_distribution.png')

In [ ]:
# sns.stripplot(x=outcome_var, y="propensity_score", hue='IVT_with_rtPA', data=data)
#

In [ ]:
# sns.stripplot(x="index", y="propensity_score", hue='IVT_with_rtPA', data=data.reset_index())



In [ ]:
# sns.displot(data.reset_index(), x="propensity_score", hue='IVT_with_rtPA', kind="kde")


